# 01_protocol.ipynb: Construction Protocol の構造化

## 学習目標
- GeoGebra の Construction Protocol（構築手順）を理解する
- Python を使って、依存関係を可視化する
- 5 段階の分解（層構造）で、構築過程を整理する

## Construction Protocol とは

GeoGebra で図形を構築すると、すべての操作が **Protocol（プロトコル）** に記録されます。

### 例: Thales の定理の構築

| # | Object | Definition | Type |
|---|--------|------------|------|
| 1 | $O$ | $(0, 0)$ | Point (Free) |
| 2 | $A$ | $(3, 0)$ | Point (Free) |
| 3 | $c$ | Circle($O$, $A$) | Circle |
| 4 | $B$ | Reflect($A$, $O$) | Point (Dependent) |
| 5 | $P$ | Point($c$) | Point (Free on Object) |
| 6 | seg1 | Segment($P$, $A$) | Line |
| 7 | seg2 | Segment($P$, $B$) | Line |
| 8 | angle | Angle($A$, $P$, $B$) | Angle |

### 特徴
- **自由な点** (Free): ユーザーが直接配置できる（$O$, $A$, $P$）
- **従属的な点** (Dependent): 他の要素から計算で決まる（$B$）
- **従属的な図形** (Derived): 他の要素から構成される（$c$, seg1, seg2, angle）

## 依存関係の有向グラフ

各オブジェクトの依存関係は、**有向グラフ** (directed graph) で表現できます。

```
O (自由)   A (自由)   c (O, A に依存)
 \         /         /
  \-------+--------/
   |              
   B (O, A に依存)  
   |
   P (c に依存) --- seg1(P, A)  seg2(P, B)
                    |           |
                    angle(A, P, B) -- seg1, seg2, A, P, B に依存
```

この有向グラフから、「何が変わると、何が影響を受けるか」が明確になります。

## 層構造（5 段階分解）

依存関係を分析すると、構築を **層** に分解できます。

### Layer 0: ユーザー入力（自由な点）
```
O = (0, 0)     # 中心（ユーザーが設定）
A = (3, 0)     # 初期点（ユーザーが設定）
```

### Layer 1: ユーザーが操作できる点
```
P = Point(c)   # 円周上の点（ユーザーがドラッグして動かせる）
```

### Layer 2: 決定的な幾何学的構造
```
c = Circle(O, A)        # O を中心に A を通る円
B = Reflect(A, O)       # O に対する A の反射
```

### Layer 3: 測定可能な属性
```
seg1 = Segment(P, A)    # 線分 PA
seg2 = Segment(P, B)    # 線分 PB
angle = Angle(A, P, B)  # 角度 ∠APB
```

### Layer 4: 検証・証明（定理の確認）
```
result = (angle ≈ 90°)  # Thales の定理の検証
```

## Python での実装

ここでは、ggblab の `parser.py` を使って、実際の Protocol を解析します。

In [ ]:
# まず、必要なライブラリをインポート
import sys
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# ggblab モジュールをインポート
from ggblab.parser import parse_construction_protocol
from ggblab.construction import load_ggb

print("Imports successful")

### 例: テスト用の Protocol DataFrame を作成

In [ ]:
# Thales の定理構成の Protocol
protocol_data = {
    'object': ['O', 'A', 'c', 'B', 'P', 'seg1', 'seg2', 'angle'],
    'definition': [
        '(0, 0)',
        '(3, 0)',
        'Circle(O, A)',
        'Reflect(A, O)',
        'Point(c)',
        'Segment(P, A)',
        'Segment(P, B)',
        'Angle(A, P, B)'
    ],
    'type': ['Point', 'Point', 'Circle', 'Point', 'Point', 'Line', 'Line', 'Angle'],
    'layer': [0, 0, 2, 2, 1, 3, 3, 3]
}

protocol_df = pd.DataFrame(protocol_data)
print("Protocol Table:")
print(protocol_df.to_string(index=False))

### 依存関係の抽出と有向グラフの構築

In [ ]:
# 有向グラフを構築
G = nx.DiGraph()

# 全オブジェクトをノードとして追加
for obj in protocol_df['object']:
    G.add_node(obj)

# 依存関係からエッジを追加
# 例: "Circle(O, A)" -> O, A から c への依存
dependencies = {
    'c': ['O', 'A'],
    'B': ['A', 'O'],
    'P': ['c'],
    'seg1': ['P', 'A'],
    'seg2': ['P', 'B'],
    'angle': ['A', 'P', 'B', 'seg1', 'seg2']
}

for target, sources in dependencies.items():
    for source in sources:
        G.add_edge(source, target)

print(f"Graph Nodes: {list(G.nodes())}")
print(f"Graph Edges: {list(G.edges())}")
print(f"Total Nodes: {G.number_of_nodes()}, Total Edges: {G.number_of_edges()}")

### グラフの可視化

In [ ]:
# グラフレイアウト
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# 図の作成
plt.figure(figsize=(12, 8))

# ノードの色を層によって分ける
node_colors = []
layer_map = {obj: layer for obj, layer in zip(protocol_df['object'], protocol_df['layer'])}
color_map = {0: 'lightblue', 1: 'lightgreen', 2: 'lightyellow', 3: 'lightcoral', 4: 'lightgray'}

for node in G.nodes():
    layer = layer_map[node]
    node_colors.append(color_map[layer])

# グラフを描画
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3000)
nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, connectionstyle='arc3,rad=0.1')

plt.title("Thales の定理: 依存関係有向グラフ", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("\nLegend:")
print("- 青: Layer 0 (自由な点)")
print("- 緑: Layer 1 (ユーザー操作可能)")
print("- 黄: Layer 2 (決定的な構造)")
print("- 赤: Layer 3 (測定属性)")
print("- 灰: Layer 4 (検証)")

### 位相ソート（トポロジカルソート）

In [ ]:
# トポロジカルソート: 依存関係を尊重した構築順序
topo_order = list(nx.topological_sort(G))

print("\nトポロジカルソート（構築可能な順序）:")
for i, obj in enumerate(topo_order, 1):
    layer = layer_map[obj]
    obj_type = protocol_df[protocol_df['object'] == obj]['type'].values[0]
    definition = protocol_df[protocol_df['object'] == obj]['definition'].values[0]
    print(f"{i:2d}. [{layer}] {obj:5s} : {obj_type:10s} = {definition}")

### 入次数 (In-Degree) と出次数 (Out-Degree)

In [ ]:
# 各ノードの入次数・出次数
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

degree_df = pd.DataFrame({
    'Object': list(G.nodes()),
    'In-Degree (依存元の数)': [in_degree[obj] for obj in G.nodes()],
    'Out-Degree (影響先の数)': [out_degree[obj] for obj in G.nodes()],
    'Type': [protocol_df[protocol_df['object'] == obj]['type'].values[0] for obj in G.nodes()]
})

print("\n入次数・出次数分析:")
print(degree_df.to_string(index=False))

print("\n解釈:")
print(f"- {protocol_df['object'].iloc[0]}: In-Degree = 0 → 全く依存していない（最初に定義）")
print(f"- angle: Out-Degree = 0 → 他に影響しない（最終的な検証）")

## 層構造の可視化

**層（Layer）の意味**:
- **Layer 0**: ユーザーが最初に設定する自由な点
- **Layer 1**: ユーザーがインタラクティブに操作できる点（ドラッグ可能）
- **Layer 2**: Layer 0, 1 から自動的に決まる幾何学的オブジェクト
- **Layer 3**: Layer 0–2 の属性を測定したもの
- **Layer 4**: Layer 3 の値から導出される検証結果

In [ ]:
# 層ごとのオブジェクト分類
layers = {0: [], 1: [], 2: [], 3: [], 4: []}

for obj, layer in zip(protocol_df['object'], protocol_df['layer']):
    layers[layer].append(obj)

print("層ごとのオブジェクト分類:")
for layer_num in sorted(layers.keys()):
    objects = layers[layer_num]
    if objects:
        print(f"\nLayer {layer_num}: {', '.join(objects)}")
        
        # 説明
        descriptions = {
            0: "ユーザーが最初に設定する点（固定 or 初期値）",
            1: "ユーザーがドラッグして動かす点",
            2: "Layer 0, 1 から決定論的に計算される図形",
            3: "図形の属性を測定したもの（距離、角度など）",
            4: "Layer 3 から導出される検証結果（真偽値など）"
        }
        print(f"説明: {descriptions[layer_num]}")

## 応用: 変更伝播（Change Propagation）

もし点 $A$ の位置を変えたら、何が影響を受けるか？

In [ ]:
def find_affected_objects(G, node):
    """
    給定ノードが変わると、どのノードが影響を受けるか計算
    （すべての後続ノード = descendants）
    """
    descendants = set()
    to_visit = [node]
    
    while to_visit:
        current = to_visit.pop()
        for successor in G.successors(current):
            if successor not in descendants:
                descendants.add(successor)
                to_visit.append(successor)
    
    return descendants

# 各オブジェクトが変わったときの影響を分析
print("変更伝播分析:")
for obj in ['O', 'A', 'P']:
    affected = find_affected_objects(G, obj)
    print(f"\n{obj} が変わると → 以下が影響を受ける: {sorted(affected)}")
    print(f"  影響を受けるオブジェクト数: {len(affected)}")

## まとめ

### 見えてきたこと
1. **Thales の定理の構成は、5 層に分解できる**
   - Layer 0: 初期設定（$O$, $A$）
   - Layer 1: ユーザー操作（$P$）
   - Layer 2: 自動計算（$c$, $B$）
   - Layer 3: 測定（seg1, seg2, angle）
   - Layer 4: 検証（結果の確認）

2. **依存関係の有向グラフから、計算順序が明確になる**
   - トポロジカルソートで実行順序を決定可能
   - 循環依存がなければ、必ず実行可能

3. **変更が自動的に伝播する**
   - $P$ を移動 → angle が自動的に再計算される
   - $A$ を移動 → $c$, $B$, angle など多くのものが影響を受ける

### 数学的な視点
- **層構造 = 認識論的レベル**（how we know）
  - 何が自由か（Layer 0, 1）
  - 何が決まるか（Layer 2）
  - 何が測定可能か（Layer 3）
  - 何が証明できるか（Layer 4）

## 思考問題

1. 「Layer 1 の点 $P$ を動かしたときだけ、angle が変わり、他は変わらないことが重要な理由は？」
2. 「もし Layer 2 に新しい構成要素を追加したら、全体にどう影響するか？」
3. 「この層構造は、Thales 以外の幾何学的定理にも適用できるか？」

## 次のステップ

次の notebook 01_verify.ipynb では、この Protocol を ggblab の **SceneVerifier** を使って、自動的に検証します。